# Chapter 9 &mdash; The GNFA: Adding `Real_I` and `Real_F` to Stand On

**Concept 1 of the Chapter 9 decomposition:** *The GNFA: Adding `Real_I` and `Real_F` to Stand On*

Wrap the NFA in one new initial and one new final state joined by $\varepsilon$ edges, so elimination has somewhere to stand.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter9-NFA2RE/Concept-The-GNFA/Concept-The-GNFA.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.Def_NFA2RE     import *
from jove.AnimateNFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateNFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateNFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


To turn an NFA into a regular expression we delete states one at a time until only two
remain, and read the label between them. For that to work we need **exactly one**
initial and **exactly one** final state, and neither may be deleted.

So wrap the machine: add **`Real_I`** with an $\varepsilon$ edge to every original
start state, and **`Real_F`** with an $\varepsilon$ edge from every original final
state. The result is a **GNFA** &mdash; a *generalized* NFA whose edges carry **regular
expressions**, not just symbols.

`Real_I` has no incoming edges and `Real_F` no outgoing ones, which is exactly what
makes them safe to stand on.

## 2. Definitions

### An NFA, and its GNFA wrapper

In [ ]:
N = md2mc('''NFA
I : 0 -> I
I : 1 -> F
F : 0 -> F
F : 1 -> I
''')
Gw = mk_gnfa(N)
print("NFA  : Q0 = %s, F = %s" % (sorted(N["Q0"]), sorted(N["F"])))
print("GNFA : Q0 = %s, F = %s" % (sorted(Gw["Q0"]), sorted(Gw["F"])))
print("GNFA states :", sorted(Gw["Q"]))

### A machine with several start and several final states

In [ ]:
Multi = md2mc('''NFA
I1 : 0 -> F1
I2 : 1 -> F2
F1 : 0 -> F1
F2 : 1 -> F2
''')
GM = mk_gnfa(Multi)

<!-- nav-strip -->

---

&larr;&nbsp;[Ch8&nbsp;12.&nbsp;Reading Off the Lengths of Strings Accepted by Any DFA](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter8-RE/Concept-Reading-Off-Lengths/Concept-Reading-Off-Lengths.ipynb) &nbsp;&middot;&nbsp; [**Chapter 9** index](https://github.com/ganeshutah/Jove/blob/master/Chapter9-NFA2RE/README.md) &nbsp;&middot;&nbsp; [Ch9&nbsp;2.&nbsp;State Elimination: Bypass Edges, Self-Loops, and the $m\times n$ Rule](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter9-NFA2RE/Concept-State-Elimination/Concept-State-Elimination.ipynb)&nbsp;&rarr;

---

## 3. Tests

The wrapper adds exactly two states.

In [ ]:
print("|Q| : NFA %d -> GNFA %d" % (len(N["Q"]), len(Gw["Q"])))
assert len(Gw["Q"]) == len(N["Q"]) + 2
assert Gw["Q0"] == {"Real_I"} and Gw["F"] == {"Real_F"}

`Real_I` has no incoming edges; `Real_F` has none outgoing. That is why they survive.

In [ ]:
into_I  = [(p, q) for (p, _), qs in Gw["Delta"].items() for q in qs if q == "Real_I"]
outof_F = [k for k in Gw["Delta"] if k[0] == "Real_F"]
print("edges into Real_I   :", into_I)
print("edges out of Real_F :", outof_F)
assert not into_I and not outof_F

Several start states collapse into one, via $\varepsilon$ edges.

In [ ]:
print("Multi : Q0 = %s, F = %s" % (sorted(Multi["Q0"]), sorted(Multi["F"])))
print("GNFA  : Q0 = %s, F = %s" % (sorted(GM["Q0"]), sorted(GM["F"])))
assert len(GM["Q0"]) == 1 and len(GM["F"]) == 1
print("\nedges from Real_I :", sorted(Edges_Exist_Via(GM, "Real_I", q) and q
                                      for q in sorted(GM["Q"]) if Edges_Exist_Via(GM, "Real_I", q)))

Wrapping does not change the language &mdash; the added edges are $\varepsilon$.

In [ ]:
_, _, restr = del_gnfa_states(mk_gnfa(N))
print("RE from the GNFA :", restr)
from itertools import product
D1 = min_dfa(nfa2dfa(N))
D2 = min_dfa(nfa2dfa(re2nfa(restr)))
print("same language as the original NFA? ", iso_dfa(D1, D2))
assert iso_dfa(D1, D2)

GNFA edges carry **regular expressions**; the original ones carry symbols.

In [ ]:
print("some GNFA edge labels :")
for k, v in list(sorted(Gw["Delta"].items()))[:6]:
    print("   %-22s -> %s" % (k, sorted(v)))

## 4. Animation

The machine before wrapping.

In [ ]:
from jove.AnimateNFA import *
AnimateNFA(N, FuseEdges=True)

## 5. Exercises


1. Wrap a machine whose start state is also final. Where do the $\varepsilon$ edges go?
2. Why can `Real_I` never be chosen for deletion?
3. What would go wrong if you reused an existing state as `Real_I`?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for all 255 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter9-NFA2RE/Concept-The-GNFA')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')